# Walmart Demand Forecasting
## Notebook 03: Feature Engineering
## Objectives
Prepare a model-ready dataset by creating meaningful features for demand forecasting.
### Dataset
- train.csv
- features.csv
- stores.csv
### Tasks
- Merge the train, features, and stores datasets.
- Create calendar-based features (Year, Month, Week).
- Generate lag features using historical sales.
- Generate rolling statistics to capture recent demand trends and volatility.
- Validate data quality after feature engineering.
- Remove rows with missing values introduced by lag and rolling features.
- Save the processed datasets for model development.
### Expected Output
By the end of this notebook, we will have a clean, model-ready dataset enriched with temporal, historical, and contextual features. This processed dataset will serve as the input for baseline forecasting models and machine learning models in the next stage of the project.

In [44]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from src.data_loader import load_data
train, features, stores = load_data()
train.head()


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [45]:
features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [46]:
stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


#1. Merging datasets

In [47]:
train = train.drop(columns=['IsHoliday'])
df = train.merge(features, on=['Store','Date'], how='left')
df = df.merge(stores, on=['Store'], how='left')
print(df.shape)
df.head()


(421570, 16)


,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday,Type,Size
0,1,1,2010-02-05,24924.50,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
1,1,1,2010-02-12,46039.49,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True,A,151315
2,1,1,2010-02-19,41595.55,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False,A,151315
3,1,1,2010-02-26,19403.54,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False,A,151315
4,1,1,2010-03-05,21827.90,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False,A,151315


In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  str    
 3   Weekly_Sales  421570 non-null  float64
 4   Temperature   421570 non-null  float64
 5   Fuel_Price    421570 non-null  float64
 6   MarkDown1     150681 non-null  float64
 7   MarkDown2     111248 non-null  float64
 8   MarkDown3     137091 non-null  float64
 9   MarkDown4     134967 non-null  float64
 10  MarkDown5     151432 non-null  float64
 11  CPI           421570 non-null  float64
 12  Unemployment  421570 non-null  float64
 13  IsHoliday     421570 non-null  bool   
 14  Type          421570 non-null  str    
 15  Size          421570 non-null  int64  
dtypes: bool(1), float64(10), int64(3), str(2)
memory usage: 48.6 MB


#2. Create time-based features

In [49]:
#Fix Date
df['Date'] = pd.to_datetime(df['Date'])

In [50]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)

- Since the dataset is aggregated at weekly level, DayOfWeek feature was excluded as it does not provide additional information.

In [51]:
#Sort data
df = df.sort_values(['Store','Dept','Date'])

#3. Setting up lag features and rolling features
lag_1: last week
lag_2: 2 weeks ago
lag_4: a month ago
rolling_mean_4: rolling trend recently (4 weeks ago)
rolling_std_4: rolling volatility recently (4 weeks ago)

In [52]:
df['lag_1'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(1)
df['lag_2'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(2)
df['lag_4'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(4)
df.head()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,...,Unemployment,IsHoliday,Type,Size,Year,Month,Week,lag_1,lag_2,lag_4
0,1,1,2010-02-05,24924.50,42.31,2.572,NaN,NaN,NaN,NaN,...,8.106,False,A,151315,2010,2,5,NaN,NaN,NaN
1,1,1,2010-02-12,46039.49,38.51,2.548,NaN,NaN,NaN,NaN,...,8.106,True,A,151315,2010,2,6,24924.50,NaN,NaN
2,1,1,2010-02-19,41595.55,39.93,2.514,NaN,NaN,NaN,NaN,...,8.106,False,A,151315,2010,2,7,46039.49,24924.50,NaN
3,1,1,2010-02-26,19403.54,46.63,2.561,NaN,NaN,NaN,NaN,...,8.106,False,A,151315,2010,2,8,41595.55,46039.49,NaN
4,1,1,2010-03-05,21827.90,46.50,2.625,NaN,NaN,NaN,NaN,...,8.106,False,A,151315,2010,3,9,19403.54,41595.55,24924.5


In [53]:
df['rolling_mean_4'] = (df.groupby(['Store','Dept'])['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(4).mean()))
df['rolling_std_4'] = (df.groupby(['Store','Dept'])['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(4).std()))

In [54]:
#Holiday feature
df['IsHoliday'] = df['IsHoliday'].astype(int)

In [55]:
#Fill MarkDown missing to 0
markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
df[markdown_cols] = df[markdown_cols].fillna(0)

#Drop NaN rows due to lag and rolling features
lag_rolling_cols = [
    "lag_1",
    "lag_2",
    "lag_4",
    "rolling_mean_4",
    "rolling_std_4"
]
df_model = df.dropna(subset=lag_rolling_cols)

In [56]:
#Check final data
print(df_model.shape)
df_model.isnull().sum()

(408436, 24)


Store             0
Dept              0
Date              0
Weekly_Sales      0
Temperature       0
Fuel_Price        0
MarkDown1         0
MarkDown2         0
MarkDown3         0
MarkDown4         0
MarkDown5         0
CPI               0
Unemployment      0
IsHoliday         0
Type              0
Size              0
Year              0
Month             0
Week              0
lag_1             0
lag_2             0
lag_4             0
rolling_mean_4    0
rolling_std_4     0
dtype: int64

In [57]:
df['Date'].min(), df_model['Date'].min()

(Timestamp('2010-02-05 00:00:00'), Timestamp('2010-03-05 00:00:00'))

In [58]:
df['Date'].max(), df_model['Date'].max()

(Timestamp('2012-10-26 00:00:00'), Timestamp('2012-10-26 00:00:00'))

Rows from the first four weeks were removed because lag and rolling features require sufficient historical observations. This is expected in time series feature engineering and helps ensure that all modeling rows contain complete historical demand information.

In [59]:
#Save data to data/processed folder
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

df.to_csv(PROCESSED_PATH / "walmart_merged.csv", index=False)
df_model.to_csv(PROCESSED_PATH / "walmart_features.csv", index=False)

### Conclusion

#### Newly Created Features

| Feature | Description | Purpose |
|---------|-------------|----------|
| Year | Extracted from Date | Capture long-term trend |
| Month | Extracted from Date | Capture seasonality |
| Week | ISO week number | Capture weekly pattern |
| lag_1 | Previous week's sales | Short-term demand dependency |
| lag_2 | Sales two weeks ago | Medium-term dependency |
| lag_4 | Sales four weeks ago | Monthly demand pattern |
| rolling_mean_4 | Average sales of previous 4 weeks | Capture recent trend |
| rolling_std_4 | Standard deviation of previous 4 weeks | Measure demand volatility |

#### Final Dataset

- Dataset successfully merged with external features and store information.
- Time-based and historical demand features were created.
- Missing values introduced by lag features were removed.
- The processed dataset is ready for model training.